# Preprocessing with SMOTE for Employee Attrition Dataset

This notebook performs the full preprocessing pipeline for the IBM HR Employee Attrition dataset.

## What this notebook does
- Loads the dataset from `../data/WA_Fn-UseC_-HR-Employee-Attrition.csv`
- Checks shape, missing values, duplicates, and target balance
- Drops unnecessary columns
- Encodes the target variable `Attrition`
- Converts categorical columns into numeric format using one-hot encoding
- Splits the dataset into training and testing sets
- Applies **SMOTE only to the training set**
- Saves all processed files into `../outputs/`


## Step 1: Import libraries

These libraries are needed for data handling, train-test splitting, and SMOTE.

In [4]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


## Step 2: Load dataset


In [5]:
file_path = '../data/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(file_path)

print('Dataset loaded successfully.')
df.head()


Dataset loaded successfully.


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## Step 3: Basic dataset inspection

This step helps you understand:
- how many rows and columns exist
- data types
- whether there are missing values
- whether duplicate rows exist

In [6]:
print('Dataset Shape:', df.shape)
print('\nColumn Names:\n', df.columns.tolist())
print('\nData Types:\n')
print(df.dtypes)

print('\nMissing Values:\n')
print(df.isnull().sum())

print('\nDuplicate Rows:', df.duplicated().sum())


Dataset Shape: (1470, 35)

Column Names:
 ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Data Types:

Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber     

## Step 4: Check target column distribution

This dataset is imbalanced. Usually, more employees belong to `No` attrition than `Yes` attrition.
That is why we later use SMOTE on the training set.

In [7]:
print('Target Variable Distribution:')
print(df['Attrition'].value_counts())

print('\nTarget Variable Distribution (Percentage):')
print(df['Attrition'].value_counts(normalize=True) * 100)


Target Variable Distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Target Variable Distribution (Percentage):
Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64


## Step 5: Drop unnecessary columns

These columns are commonly removed in this dataset:
- `EmployeeCount` → constant value
- `Over18` → constant value
- `StandardHours` → constant value
- `EmployeeNumber` → ID-like column, not useful for prediction

In [8]:
columns_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df = df.drop(columns=columns_to_drop)

print('Shape after dropping unnecessary columns:', df.shape)
df.head()


Shape after dropping unnecessary columns: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


## Step 6: Encode target column

Machine learning models require numeric target values.

We convert:
- `Yes` → 1
- `No` → 0

In [9]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

print('Encoded target values:')
print(df['Attrition'].value_counts())


Encoded target values:
Attrition
0    1233
1     237
Name: count, dtype: int64


## Step 7: Separate features and target

- `X` contains input features
- `y` contains the target column `Attrition`

In [10]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)


Feature matrix shape: (1470, 30)
Target vector shape: (1470,)


## Step 8: One-hot encode categorical features

SMOTE works only with numeric data. Since this dataset contains categorical columns,
we convert them into numeric columns using `pd.get_dummies()`.

`drop_first=True` is used to avoid unnecessary duplicate dummy columns.

In [11]:
X = pd.get_dummies(X, drop_first=True)

print('Shape after one-hot encoding:', X.shape)
X.head()


Shape after one-hot encoding: (1470, 44)


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,41,1102,1,2,2,94,3,2,4,5993,...,False,False,False,False,False,True,False,False,True,True
1,49,279,8,1,3,61,2,2,2,5130,...,False,False,False,False,True,False,False,True,False,False
2,37,1373,2,2,4,92,2,1,3,2090,...,True,False,False,False,False,False,False,False,True,True
3,33,1392,3,4,4,56,3,1,3,2909,...,False,False,False,False,True,False,False,True,False,True
4,27,591,2,1,1,40,3,1,2,3468,...,True,False,False,False,False,False,False,True,False,False


## Step 9: Train-test split

We split the data before SMOTE.

`stratify=y` keeps the class ratio similar in both training and testing sets.
This is important for imbalanced datasets.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape :', y_test.shape)

print('\nTraining target distribution before SMOTE:')
print(y_train.value_counts())

print('\nTesting target distribution:')
print(y_test.value_counts())


X_train shape: (1176, 44)
X_test shape : (294, 44)
y_train shape: (1176,)
y_test shape : (294,)

Training target distribution before SMOTE:
Attrition
0    986
1    190
Name: count, dtype: int64

Testing target distribution:
Attrition
0    247
1     47
Name: count, dtype: int64


## Step 10: Apply SMOTE only to training data

This is the most important part.

SMOTE creates synthetic minority-class samples to balance the training data.
We apply it only to the training set to avoid data leakage.

In [14]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Training target distribution before SMOTE:')
print(y_train.value_counts())

print('\nTraining target distribution after SMOTE:')
print(pd.Series(y_train_smote).value_counts())


Training target distribution before SMOTE:
Attrition
0    986
1    190
Name: count, dtype: int64

Training target distribution after SMOTE:
Attrition
0    986
1    986
Name: count, dtype: int64


## Step 11: Create output folder

This ensures the `../outputs/` folder exists before saving files.

In [15]:
os.makedirs('../outputs', exist_ok=True)
print('outputs folder is ready')


outputs folder is ready


## Step 12: Save processed files

Files saved:
- `preprocessed_data.csv` → cleaned full dataset
- `X_train_smote.csv` → training features after SMOTE
- `y_train_smote.csv` → training labels after SMOTE
- `X_test.csv` → original test features
- `y_test.csv` → original test labels

In [16]:
df.to_csv('../outputs/preprocessed_data.csv', index=False)
X_train_smote.to_csv('../outputs/X_train_smote.csv', index=False)
pd.DataFrame(y_train_smote, columns=['Attrition']).to_csv('../outputs/y_train_smote.csv', index=False)
X_test.to_csv('../outputs/X_test.csv', index=False)
pd.DataFrame(y_test, columns=['Attrition']).to_csv('../outputs/y_test.csv', index=False)

print('All files saved successfully in ../outputs/')


All files saved successfully in ../outputs/


## Step 13: Verify saved files

This final step confirms that all output files were created correctly.

In [17]:
print('Saved files:')
print(os.listdir('../outputs'))


Saved files:
['preprocessed_data.csv', 'X_test.csv', 'X_train_smote.csv', 'y_test.csv', 'y_train_smote.csv']
